# 8.24 Summarization

Summarization is compression with a conscience: fewer words, but the important facts must survive.

Summarization lives between selection and generation. It must decide what matters, avoid repetition, and compress without dropping or inventing important source facts.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def summarization_ladder():
    return [
        {"name": "D1 lesson sentence/fact toy", "sentences": ["a b c", "a b repeat", "d e"], "salience": np.array([0.9, 0.8, 0.7]), "sim": np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.2], [0.1, 0.2, 1.0]]), "facts": [{"a", "b", "c"}, {"a", "b"}, {"d", "e"}], "reference": {"a", "b", "c", "d", "e"}, "budget": 2, "difficulty": 1},
        {"name": "D2 clean mini-docs", "sentences": ["product launched", "users adopted", "revenue rose"], "salience": np.array([0.8, 0.7, 0.6]), "sim": np.array([[1.0, 0.1, 0.1], [0.1, 1.0, 0.2], [0.1, 0.2, 1.0]]), "facts": [{"launch"}, {"adoption"}, {"revenue"}], "reference": {"launch", "adoption", "revenue"}, "budget": 2, "difficulty": 2},
        {"name": "D3 redundancy and near-equal salience", "sentences": ["storm delayed flights", "flights delayed by storm", "crews reopened gates", "travelers got refunds"], "salience": np.array([0.9, 0.88, 0.7, 0.6]), "sim": np.array([[1.0, 0.9, 0.2, 0.1], [0.9, 1.0, 0.2, 0.1], [0.2, 0.2, 1.0, 0.2], [0.1, 0.1, 0.2, 1.0]]), "facts": [{"storm", "delay"}, {"storm", "delay"}, {"reopen"}, {"refund"}], "reference": {"storm", "delay", "reopen", "refund"}, "budget": 2, "difficulty": 4},
        {"name": "D4 news and support docs", "sentences": ["ticket failed login", "password reset fixed login", "agent escalated billing", "customer confirmed access"], "salience": np.array([0.85, 0.8, 0.65, 0.7]), "sim": np.array([[1.0, 0.7, 0.1, 0.3], [0.7, 1.0, 0.1, 0.3], [0.1, 0.1, 1.0, 0.2], [0.3, 0.3, 0.2, 1.0]]), "facts": [{"login", "failure"}, {"reset", "login"}, {"billing"}, {"access"}], "reference": {"login", "failure", "reset", "access"}, "budget": 2, "difficulty": 6},
        {"name": "D5 omitted unsupported facts", "sentences": ["doctor changed dosage", "patient reported dizziness", "patient reported dizziness again", "followup scheduled Friday", "summary claims surgery planned"], "salience": np.array([0.88, 0.86, 0.84, 0.7, 0.9]), "sim": np.array([[1.0, 0.2, 0.2, 0.1, 0.1], [0.2, 1.0, 0.95, 0.1, 0.1], [0.2, 0.95, 1.0, 0.1, 0.1], [0.1, 0.1, 0.1, 1.0, 0.1], [0.1, 0.1, 0.1, 0.1, 1.0]]), "facts": [{"dosage"}, {"dizziness"}, {"dizziness"}, {"followup"}, {"surgery"}], "reference": {"dosage", "dizziness", "followup"}, "source_facts": {"dosage", "dizziness", "followup"}, "budget": 2, "difficulty": 9},
    ]


def fact_recall(selected_facts, reference):
    if not reference:
        return 0.0
    return len(selected_facts & reference) / len(reference)


## The concept, built once (D1)\n\nMaximal marginal relevance trades salience against redundancy, and ROUGE/fact recall checks retained content: $$\operatorname{MMR}(s)=\lambda\operatorname{sal}(s)-(1-\lambda)\max_{c\in C}\operatorname{sim}(s,c),\qquad \operatorname{ROUGE_R}=\frac{|\text{reference facts}\cap\text{summary facts}|}{|\text{reference facts}|}$$\n\nThe lesson toy has salience margin 0.1, MMR scores `[0.33,0.32,0.46]`, compression ratio 0.167, and fact recall 0.6.

In [ ]:
salience = np.array([0.9, 0.4, 0.8, 0.2])
assert float(salience.max()) == 0.9
assert round(float(salience.max() - sorted(salience)[-2]), 1) == 0.1
lesson_salience = np.array([0.9, 0.8, 0.7])
lesson_similarity = np.array([1.0, 0.8, 0.1])
mmr_scores = 0.7 * lesson_salience - 0.3 * lesson_similarity
assert [round(float(x), 2) for x in mmr_scores] == [0.33, 0.32, 0.46]
assert 0.46 > 0.32
ratios = [20 / 120, 40 / 120, 80 / 120]
assert [round(x, 3) for x in ratios] == [0.167, 0.333, 0.667]
assert fact_recall({"a", "b", "c"}, {"a", "b", "c", "d", "e"}) == 0.6
source = [1, 1, 0, 1]
summary = [1, 0, 0, 1]
assert sum(source) == 3
assert round(sum(summary) / sum(source), 3) == 0.667
print("Lesson numbers verified")

Now build one reusable MMR summarizer. It selects the first high-salience sentence, then penalizes redundancy and checks selected facts against source facts.

In [ ]:
def mmr_summarize(sentences, salience, sim, lambda_=0.7, budget=2):
    selected = []
    while len(selected) < budget:
        best = None
        for idx in range(len(sentences)):
            if idx in selected:
                continue
            redundancy = 0.0
            if selected:
                redundancy = max(float(sim[idx, chosen]) for chosen in selected)
            score = lambda_ * float(salience[idx]) - (1 - lambda_) * redundancy
            candidate = (score, idx)
            if best is None or candidate > best:
                best = candidate
        selected.append(best[1])
    return selected

rung = summarization_ladder()[0]
selected = mmr_summarize(rung["sentences"], rung["salience"], rung["sim"], budget=rung["budget"])
selected_facts = set().union(*(rung["facts"][idx] for idx in selected))
assert selected == [0, 2]
assert fact_recall(selected_facts, rung["reference"]) == 1.0
print(selected, selected_facts)

## The dataset ladder

The gap topic gets an inline D1–D5 ladder: a lesson fact toy, clean mini-docs, redundancy/near-ties, news/support documents, and longer docs with omitted or unsupported facts.

In [ ]:
rungs = summarization_ladder()
for rung in rungs:
    print(rung["name"], "sentences=", len(rung["sentences"]), "budget=", rung["budget"], "reference_facts=", len(rung["reference"]))
    print("sample", rung["sentences"][:2])

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    selected = mmr_summarize(rung["sentences"], rung["salience"], rung["sim"], budget=rung["budget"])
    selected_facts = set().union(*(rung["facts"][idx] for idx in selected))
    supported = selected_facts & rung.get("source_facts", selected_facts)
    recall = fact_recall(supported, rung["reference"])
    rows.append((rung["name"], rung["difficulty"], recall, selected, supported))
for name, difficulty, recall, selected, supported in rows:
    print(f"{name:34s} difficulty={difficulty:2d} recall={recall:.3f} selected={selected} facts={sorted(supported)}")

## Results visualization

In [ ]:
fig, axes = plt.subplots(1, len(rungs), figsize=(16, 3))
metrics = []
difficulties = []
for col, rung in enumerate(rungs):
    selected = mmr_summarize(rung["sentences"], rung["salience"], rung["sim"], budget=rung["budget"])
    selected_facts = set().union(*(rung["facts"][idx] for idx in selected))
    supported = selected_facts & rung.get("source_facts", selected_facts)
    metrics.append(fact_recall(supported, rung["reference"]))
    difficulties.append(rung["difficulty"])
    axes[col].imshow(rung["sim"], cmap="Blues", vmin=0, vmax=1)
    axes[col].set_title(rung["name"].split()[0])
    axes[col].set_xlabel("sentence")
    axes[col].set_ylabel("sentence")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(difficulties, metrics, marker="o")
plt.xlabel("compression / document difficulty")
plt.ylabel("fact recall")
plt.title("Summary recall vs difficulty")
plt.ylim(0, 1.05)
plt.show()

## Pitfall on the hardest rung

Salience-only selection repeats information or even picks unsupported claims. The fix uses MMR for diversity and source fact checks before scoring recall.

In [ ]:
hard = rungs[-1]
salience_only = list(np.argsort(hard["salience"])[::-1][:hard["budget"]])
salience_facts = set().union(*(hard["facts"][idx] for idx in salience_only))
salience_supported = salience_facts & hard["source_facts"]
mmr_selected = mmr_summarize(hard["sentences"], hard["salience"], hard["sim"], budget=hard["budget"])
mmr_facts = set().union(*(hard["facts"][idx] for idx in mmr_selected))
mmr_supported = mmr_facts & hard["source_facts"]
print("salience only", salience_only, salience_facts, "recall", fact_recall(salience_supported, hard["reference"]))
print("MMR + fact check", mmr_selected, mmr_supported, "recall", fact_recall(mmr_supported, hard["reference"]))
assert "surgery" in salience_facts
assert "surgery" not in mmr_supported
assert fact_recall(mmr_supported, hard["reference"]) >= fact_recall(salience_supported, hard["reference"])

## Evaluate it + Practice

- Metric: ROUGE/fact recall under a fixed compression budget.
- No-skill baseline: take top-salience sentences only.
- Cheap sanity check: selected facts are a subset of source-supported facts.
- Ablation: turn off redundancy penalty and repeated evidence crowds out new facts.
- Failure signals: short summaries omit reference facts or abstraction invents unsupported glue.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.